# Feature preprocessing — librosa grid and Beat This! grid

Generates both training datasets for the grid-ablation study from
`data/audio/processed/{SongID}.mp3` and `data/clean_labeled.csv`:

| Pass | Meter grid | Output |
|---|---|---|
| 1 | librosa tempo extrapolation (original) | `data/seg_librosa/`, `data/lab_librosa/` |
| 2 | Beat This! downbeats | `data/seg_beatthis/`, `data/lab_beatthis/` |

Both passes share the identical feature pipeline (RMS + mel/chroma/tempogram/MFCC
NMF activations, standardized, dimension-weighted, hierarchical positional
encoding) and differ only in how the meter grid is built. The four NMF fits per
song run on the GPU via `pytorch_core.nmf_torch` — a torch port of sklearn's
default NMF (NNDSVDA init, HALS coordinate descent, Frobenius loss) that
reproduces librosa's component sorting.

This is the notebook counterpart of `scripts/preprocess.py`; for bulk CPU-parallel
runs the CLI with `--song-ids` chunking remains the faster option. Passes here run
serially and are resumable: songs whose pickles already exist are skipped, so an
interrupted run continues where it left off.

In [ ]:
import os
import sys
import time
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location("preprocess", REPO_ROOT / "scripts" / "preprocess.py")
pp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pp)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Configuration ---------------------------------------------------------
CSV_PATH = REPO_ROOT / "data" / "clean_labeled.csv"
AUDIO_DIR = REPO_ROOT / "data" / "audio" / "processed"
SPLITS_DIR = REPO_ROOT / "data" / "splits"
NMF_DEVICE = DEVICE          # torch NMF device; set to None to fall back to sklearn
INCLUDE_TEST = False         # training needs train+val only; flip on to also build test pickles
LIMIT = None                 # e.g. 3 for a smoke run

PASSES = {
    "librosa":   dict(grid_source="librosa",   segments_dir="data/seg_librosa",  labels_dir="data/lab_librosa"),
    "beat_this": dict(grid_source="beat_this", segments_dir="data/seg_beatthis", labels_dir="data/lab_beatthis"),
}

df = pd.read_csv(CSV_PATH)


def read_ids(name):
    return [int(line) for line in (SPLITS_DIR / f"{name}_songs.txt").read_text().split()]


song_ids = read_ids("train") + read_ids("val") + (read_ids("test") if INCLUDE_TEST else [])
song_ids = [s for s in song_ids if s in set(df["SongID"].unique())]
if LIMIT:
    song_ids = song_ids[:LIMIT]

print(f"device={DEVICE}  songs={len(song_ids)}  include_test={INCLUDE_TEST}")

## GPU NMF equivalence check

Before trusting the torch NMF for dataset generation, verify on one real song
that it matches the sklearn reference: same relative reconstruction error and
near-perfect correlation between each torch activation and its sklearn
counterpart. Run on the largest matrix in the pipeline (the 128-band mel
spectrogram) and the widest one (the 384-lag tempogram).

In [ ]:
import librosa
from pytorch_core.nmf_torch import decompose

sid = song_ids[0]
y, sr = librosa.load(str(AUDIO_DIR / f"{sid}.mp3"), sr=pp.TARGET_SR)
_, y_perc = librosa.effects.hpss(y)

mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=pp.HOP_LENGTH)
onset_env = librosa.onset.onset_strength(y=y_perc, sr=sr, hop_length=pp.HOP_LENGTH)
tempogram = np.clip(librosa.feature.tempogram(onset_envelope=onset_env, sr=sr,
                                              hop_length=pp.HOP_LENGTH), 0, None)

decompose(mel, 3, device=NMF_DEVICE)  # warmup (CUDA context + kernel compilation)

for name, V, k in [("mel", mel, 3), ("tempogram", tempogram, 3)]:
    t0 = time.time(); c_ref, a_ref = librosa.decompose.decompose(V, n_components=k, sort=True); t_cpu = time.time() - t0
    t0 = time.time(); c, a = decompose(V, k, device=NMF_DEVICE); t_gpu = time.time() - t0
    err_ref = np.linalg.norm(V - c_ref @ a_ref) / np.linalg.norm(V)
    err = np.linalg.norm(V - c @ a) / np.linalg.norm(V)
    corrs = [max(abs(np.corrcoef(a[i], a_ref[j])[0, 1]) for j in range(k)) for i in range(k)]
    print(f"{name:10s} {V.shape}  sklearn {t_cpu:5.2f}s err {err_ref:.4f} | "
          f"torch {t_gpu:5.2f}s err {err:.4f} | corr {[round(float(x), 3) for x in corrs]}")

## Shared pass runner

Wraps `scripts/preprocess.py::process_song` — the exact function the CLI uses —
so the notebook and CLI produce byte-identical pickles for a given grid and NMF
backend. Existing pickles are skipped (delete the output dirs to force a rebuild).

In [ ]:
def run_pass(name, grid_source, segments_dir, labels_dir):
    seg_dir, lab_dir = REPO_ROOT / segments_dir, REPO_ROOT / labels_dir
    seg_dir.mkdir(parents=True, exist_ok=True)
    lab_dir.mkdir(parents=True, exist_ok=True)

    processed, skipped, failed = 0, 0, []
    max_meters, max_frames = 0, 0
    t_start = time.time()

    for sid in tqdm(song_ids, desc=f"{name} grid"):
        if (seg_dir / f"{sid}_data.pkl").exists() and (lab_dir / f"{sid}_labels.pkl").exists():
            skipped += 1
            continue
        audio_path = AUDIO_DIR / f"{sid}.mp3"
        if not audio_path.exists():
            failed.append((sid, "audio file not found"))
            continue
        try:
            n_meters, n_frames = pp.process_song(
                sid, df.loc[df["SongID"] == sid], str(audio_path),
                str(seg_dir), str(lab_dir),
                grid_source=grid_source, device=DEVICE, nmf_device=NMF_DEVICE)
            max_meters, max_frames = max(max_meters, n_meters), max(max_frames, n_frames)
            processed += 1
        except Exception as e:
            failed.append((sid, f"{type(e).__name__}: {e}"))

    elapsed = time.time() - t_start
    per_song = elapsed / processed if processed else 0.0
    print(f"\n[{name}] processed={processed} skipped={skipped} failed={len(failed)} "
          f"({elapsed / 60:.1f} min, {per_song:.0f}s/song)")
    if processed:
        print(f"[{name}] max meters/song={max_meters}, max frames/meter={max_frames} "
              f"(dataset pads to MAX_METERS=201, MAX_FRAMES=300)")
    for sid, reason in failed:
        print(f"  FAILED {sid}: {reason}")
    return failed

## Pass 1 — librosa meter grid

Original grid: extrapolates the CSV tempo (Spotify metadata, librosa fallback)
from the first detected beat and takes every `time_signature`-th beat as a
meter boundary. Assumes constant tempo and 4/4 throughout the song.

In [ ]:
failed_librosa = run_pass("librosa", **PASSES["librosa"])

## Pass 2 — Beat This! downbeat grid

Meter boundaries come from downbeats tracked by the Beat This! neural tracker
(`pytorch_core/downbeats.py`), so the grid follows the song's actual metrical
structure — tempo drift, dropped beats, and non-4/4 sections included. The
tracker runs on the same device as the NMF fits.

In [ ]:
failed_beatthis = run_pass("beat_this", **PASSES["beat_this"])

## Summary

Pickle counts per dataset. Both passes must cover the same songs before the
grid-ablation trials (`notebooks/Grid-Ablation-Trials.ipynb`) can compare the
two grids fairly.

In [ ]:
for name, cfg in PASSES.items():
    seg_dir = REPO_ROOT / cfg["segments_dir"]
    lab_dir = REPO_ROOT / cfg["labels_dir"]
    n_seg = len(list(seg_dir.glob("*_data.pkl"))) if seg_dir.exists() else 0
    n_lab = len(list(lab_dir.glob("*_labels.pkl"))) if lab_dir.exists() else 0
    print(f"{name:10s} segments={n_seg:4d}  labels={n_lab:4d}  (target {len(song_ids)})")

only_librosa = set(p.stem.split("_")[0] for p in (REPO_ROOT / PASSES["librosa"]["segments_dir"]).glob("*_data.pkl"))
only_beatthis = set(p.stem.split("_")[0] for p in (REPO_ROOT / PASSES["beat_this"]["segments_dir"]).glob("*_data.pkl"))
diff = only_librosa ^ only_beatthis
if diff:
    print(f"coverage mismatch ({len(diff)} songs): {sorted(diff)[:20]}")
else:
    print("both datasets cover the same songs")